# 🧪 Experimento: TODO Aguascalientes "calidad Landsat" en el navegador

6 bandas, 30 m/px (3696×3335 = 12.3 Mpx). La pregunta: ¿cabe un estado
completo dentro del límite de 4 GB de WebAssembly de 32 bits?

In [ ]:
%pip install -q shepherd-wasm
import platform, time
import numpy as np
import scipy.ndimage, sklearn.cluster
import shepherd_wasm

def heap_mb():
    try:
        import pyodide_js
        return f"{pyodide_js._module.HEAPU8.length/1e6:.0f} MB"
    except Exception:
        return "n/d"

print(f"Arquitectura: '{platform.machine()}'  |  heap wasm inicial: {heap_mb()}")

In [ ]:
t0 = time.time()
import js
from pyodide.http import pyfetch
r = await pyfetch("/mis_datos/ags_landsat_30m.tif")
datos = await r.bytes()
open("/tmp/ags_landsat_30m.tif", "wb").write(datos)
# /tmp (disco del kernel): escribir en la carpeta del cuaderno
# empuja 176 MB por el service worker (lentisimo y fragil)
del datos
print(f"descarga local: {time.time()-t0:.1f} s  |  heap: {heap_mb()}")

In [ ]:
import rasterio
t0 = time.time()
with rasterio.open("/tmp/ags_landsat_30m.tif") as src:
    img = src.read().astype(np.float32)
    descripciones = src.descriptions
print(f"imagen estatal cargada: {img.shape} en {time.time()-t0:.1f} s")
print(f"bandas: {descripciones}")
print(f"heap wasm: {heap_mb()}")

In [ ]:
t0 = time.time()
res = shepherd_wasm.doShepherdSegmentation(
    img, numClusters=60, minSegmentSize=50, maxSpectralDiff="auto",
    imgNullVal=-9999, fixedKMeansInit=True)
seg = res.segimg
nseg = int(seg.max())
print(f"segmentacion estatal: {nseg:,} segmentos en {time.time()-t0:.0f} s")
print(f"heap wasm: {heap_mb()}")

In [ ]:
sizes = np.bincount(seg.ravel())[1:]
flat = seg.ravel()
medias = np.empty((nseg, img.shape[0]), dtype=np.float32)
for b in range(img.shape[0]):
    medias[:, b] = (np.bincount(flat, weights=img[b].ravel())[1:]
                    / np.maximum(sizes, 1))
print(f"tabla de objetos: {medias.shape[0]:,} segmentos x {medias.shape[1]} bandas")
print(f"segmento mas grande: {sizes.max():,} px = {sizes.max()*900/1e6:.1f} km2")
print(f"heap wasm: {heap_mb()}")

In [ ]:
import matplotlib.pyplot as plt
rgb = np.stack([img[2], img[1], img[0]], axis=-1)[::4, ::4]
p2, p98 = np.percentile(rgb, (2, 98))
rgb = np.clip((rgb - p2) / (p98 - p2), 0, 1)
fig, ax = plt.subplots(1, 2, figsize=(11, 5))
ax[0].imshow(rgb); ax[0].set_title("Aguascalientes (RGB 30 m)"); ax[0].axis("off")
ax[1].imshow(np.where(seg[::4, ::4] % 97 == 0, 1, np.nan), cmap="autumn")
ax[1].imshow(rgb, alpha=0.6); ax[1].set_title(f"{nseg:,} segmentos"); ax[1].axis("off")
plt.tight_layout(); plt.show()
print(f"heap wasm final: {heap_mb()}")
print("EXPERIMENTO LANDSAT AGS: COMPLETO")